# 1 · Chain of Thought — Show The Working

**The problem:** an AI model starts writing its answer immediately. It has no rough paper.
If a question needs three steps of working and you only ask for the final answer,
those three steps never happen — and the answer is often wrong, while *sounding* confident.

**The fix:** make the model write the working **before** the answer. That is all
Chain of Thought is.

Two examples in this notebook:

1. Ordering materials for a training workshop (a calculation)
2. Reading customer feedback (a judgement)

One rule to watch for in both: **the answer always comes last.**

In [ ]:
# ---- Step 0: check the kernel, then install what is missing ----
# Run this first. It works on Colab, on a fresh laptop,
# and it tells you plainly if the notebook is running the wrong Python.

import importlib.util, subprocess, sys

if sys.version_info < (3, 10):
    print("STOP - this notebook needs Python 3.10 or newer.")
    print("This kernel is Python", sys.version.split()[0], "at", sys.executable)
    print()
    print("Fix it like this:")
    print("  In Jupyter / VS Code : Kernel > Change Kernel, and pick the one from")
    print("                         structured_prompting/.venv")
    print("  On Colab             : Runtime > Restart session, then run this cell again")
    raise SystemExit("Wrong Python version - see the message above.")

REQUIRED = [
    ("openai", "openai==2.53.0"),
    ("dotenv", "python-dotenv==1.2.2"),
]

missing = [pkg for mod, pkg in REQUIRED if importlib.util.find_spec(mod) is None]

if missing:
    print("Installing:", ", ".join(missing))
    print("(a minute the first time, nothing the next time)")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
    print("Done.")
else:
    print("All libraries already here.")

print("Python", sys.version.split()[0], "at", sys.executable)


In [ ]:
# ---- Setup: run this cell first ----
# Loads your OpenAI key from the .env file and creates one helper function.

import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()                                   # reads .env in this folder
if not os.getenv("OPENAI_API_KEY"):
    try:
        # Google Colab: add OPENAI_API_KEY in the Secrets panel (the key icon, left)
        from google.colab import userdata
        os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    except Exception:
        import getpass  # last resort: type it here, it is not saved anywhere
        os.environ["OPENAI_API_KEY"] = getpass.getpass("Paste your OpenAI key: ")

client = OpenAI()                               # uses OPENAI_API_KEY automatically
MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")

def ask(prompt, temperature=0):
    """Send one prompt to the model and return its reply as plain text."""
    reply = client.chat.completions.create(
        model=MODEL,
        temperature=temperature,
        messages=[{"role": "user", "content": prompt}],
    )
    return reply.choices[0].message.content

print("Setup done. Using model:", MODEL)

---
## Example 1 · Ordering materials for a training workshop

**Situation:** 45 participants are confirmed for next week's workshop.
Each participant needs 3 printed booklets. The print shop sells booklets
in packs of 20, and 1 full pack is already in the store room.

**We want:** how many packs to order.

**The correct answer:** 45 × 3 = 135 booklets → 135 ÷ 20 = 6.75, so 7 packs
are needed → 1 already in stock → **order 6 packs**.

### The wrong approach — ask for just the answer

In [ ]:
wrong_prompt = """45 participants are confirmed for a workshop.
Each participant needs 3 printed booklets.
Booklets come in packs of 20. 1 pack is already in stock.

How many packs should we order? Give me just the number, nothing else."""

print(ask(wrong_prompt))

Run it a few times. Sometimes it says 6, sometimes 7.
And even when the number is right, **you cannot check it** — there is no working
to look at. If this were a purchase order for 4,500 laptops instead of booklets,
"just the number" would be a very expensive habit.

### The right approach — name the steps, answer last

In [ ]:
right_prompt = """45 participants are confirmed for a workshop.
Each participant needs 3 printed booklets.
Booklets come in packs of 20. 1 pack is already in stock.

Work it out in this order, one line each:
1. Total booklets needed
2. Packs needed (round up to a whole pack)
3. Subtract the packs already in stock

Then write the final number on its own line, starting with ANSWER:"""

print(ask(right_prompt))

Same model, same question. We only gave it space to write the middle numbers.

Now every line can be checked by a human — and each line uses the line above it,
which is what makes the final number reliable.

> **Lesson:** the words the model writes are its rough paper.
> No room for the working means there was no working.

---
## Example 2 · Reading customer feedback

**Situation:** your team collects customer feedback. This message arrives:

> *"Finally someone replied to my complaint, after three days of waiting."*

**We want:** is this customer satisfied or dissatisfied?
(Read it yourself: they are clearly **dissatisfied** — "finally" here is a complaint
about the delay, not praise.)

**Why it is hard:** the word "finally" *looks* positive. If the model answers
before it thinks, it will grab that word and get it wrong.

### The wrong approach — verdict first

In [ ]:
feedback = "Finally someone replied to my complaint, after three days of waiting."

wrong_prompt = f"""Here is a customer message:

"{feedback}"

Is this customer satisfied or dissatisfied? Answer with one word first,
then explain your reasoning."""

print(ask(wrong_prompt))

The verdict is written **before** any thinking happens, so the explanation
afterwards is just an excuse for a decision already made. It often says
"satisfied" because "finally" sounds positive.

### The right approach — quote, explain, then verdict

In [ ]:
right_prompt = f"""Here is a customer message:

"{feedback}"

Answer in this order:
1. QUOTE: copy the exact words that reveal the customer's feeling
2. MEANING: explain in one line what those words tell us
3. VERDICT: only now, one word — SATISFIED or DISSATISFIED"""

print(ask(right_prompt))

Exactly the same question. We only moved the verdict to the **end**.
Now the quote and the explanation are already on the page when the verdict is
written — so they actually influence it.

> **Lesson:** always put the answer last. In a prompt, in a report, anywhere.
> If the conclusion comes before the reasoning, the reasoning is decoration.

---
## Summary

1. **What it is:** make the model write the middle steps before the final answer.
2. **How:** number the steps yourself, and ask for the answer on its own last line.
3. **The one rule:** the answer always comes last.
4. **Try it yourself:** change the numbers in Example 1 (say 62 participants,
   packs of 25, 2 packs in stock) and check the model's working by hand.